# SecondSite: Cross-Hospital Cardiovascular Triage with Routine Checkup Calibration and Uncertainty Abstention
**Track:** AI + Health | **Team:** Varun Bajpai, Laziza Farrukhovna, Zainab Jamshed

> *This project is a prototype and is not medical advice.*

In [1]:
import json, pandas as pd
from data_prep import load_clean, cohort_summary, missingness_by_hospital, missingness_leak_probe

data = load_clean()
print(f'Clean patient records across 4 hospitals: {len(data)}')
display(cohort_summary(data))
display(missingness_by_hospital(data))
display(missingness_leak_probe(data))

Clean patient records across 4 hospitals: 918


,patients,disease,women,median_age,prevalence_%
hospital,,,,,
cleveland,303,139,97,56.0,45.9
hungary,293,106,80,49.0,36.2
switzerland,123,115,10,56.0,93.5
long_beach_va,199,148,6,60.0,74.4


hospital,group,cleveland,hungary,switzerland,long_beach_va
age,checkup,0.0,0.0,0.0,0.0
sex,checkup,0.0,0.0,0.0,0.0
cp,checkup,0.0,0.0,0.0,0.0
trestbps,checkup,0.0,0.3,1.6,28.6
chol,checkup,0.0,7.5,100.0,28.1
fbs,checkup,0.0,2.7,61.0,3.5
restecg,checkup,0.0,0.3,0.8,0.0
thalach,workup,0.0,0.3,0.8,26.6
exang,workup,0.0,0.3,0.8,26.6
oldpeak,workup,0.0,0.0,4.9,28.1


,question,metric,flags_only,no_information
0,Guess the hospital from missing-value flags alone,accuracy (5-fold),0.845,0.33
1,Guess heart disease from missing-value flags a...,ROC-AUC (5-fold),0.745,0.50


In [2]:
t01 = json.load(open('results/tier0_tier1.json'))
print('=== TIER 0 & TIER 1 (CLEVELAND HOLD-OUT) ===')
display(pd.DataFrame(t01['cleveland_test']))
print('=== SAME 13-FEATURE MODEL AT UNSEEN HOSPITALS ===')
display(pd.DataFrame(t01['other_hospitals']))

=== TIER 0 & TIER 1 (CLEVELAND HOLD-OUT) ===


,model,n_test,accuracy,recall_sensitivity,specificity,missed_cases_FN,false_alarms_FP,true_positives,true_negatives
0,Tier 0: majority class,60,0.533,0.000,1.000,28,0,0,32
1,Tier 1: logistic regression,60,0.850,0.821,0.875,5,4,23,28


=== SAME 13-FEATURE MODEL AT UNSEEN HOSPITALS ===


,model,n_test,accuracy,recall_sensitivity,specificity,missed_cases_FN,false_alarms_FP,true_positives,true_negatives,workup_values_imputed_%
0,Tier 0 @ hungary,293,0.638,0.000,1.000,106,0,0,187,42.4
1,Tier 1 @ hungary,293,0.819,0.566,0.963,46,7,60,180,42.4
2,Tier 0 @ switzerland,123,0.065,0.000,1.000,115,0,0,8,26.4
3,Tier 1 @ switzerland,123,0.537,0.513,0.875,56,1,59,7,26.4
4,Tier 0 @ long_beach_va,199,0.256,0.000,1.000,148,0,0,51,52.3
5,Tier 1 @ long_beach_va,199,0.543,0.453,0.804,81,10,67,41,52.3


In [3]:
t3 = json.load(open('results/tier3.json'))
print('=== THRESHOLDS CHOSEN ON CLEVELAND OOF ===')
print(t3['thresholds'])
print('=== TIER 3 CHECKUP-ONLY TRIAGE RESULTS ===')
display(pd.DataFrame(t3['triage']))
print('=== SEX & AGE SUBGROUP FAIRNESS AUDIT ===')
display(pd.read_csv('results/fairness_audit.csv'))

=== THRESHOLDS CHOSEN ON CLEVELAND OOF ===
{'rule_out_shared': 0.2912, 'rule_out_women': 0.1144, 'rule_out_men': 0.33, 'refer': 0.6319, 'target_sensitivity': 0.9, 'target_referral_precision': 0.8, 'notes': ['women: own rule-out line from 25 disease cases', 'men: own rule-out line from 114 disease cases']}
=== TIER 3 CHECKUP-ONLY TRIAGE RESULTS ===


,model,n,disease_cases,sent_routine,sent_stress_test,sent_referral,stress_test_rate,missed_cases_FN,false_alarms_FP,sensitivity,sensitivity_95ci,specificity,specificity_95ci,referral_precision
0,shared line @ cleveland (OOF),303,139,103,99,101,0.327,13,74,0.906,"[0.856, 0.95]",0.549,"[0.47, 0.622]",0.802
1,sex-specific lines @ cleveland (OOF),303,139,75,127,101,0.419,14,103,0.899,"[0.849, 0.95]",0.372,"[0.299, 0.445]",0.802
2,shared line @ hungary,293,106,144,76,73,0.259,14,57,0.868,"[0.792, 0.934]",0.695,"[0.626, 0.759]",0.753
3,sex-specific lines @ hungary,293,106,131,89,73,0.304,13,69,0.877,"[0.811, 0.934]",0.631,"[0.556, 0.701]",0.753
4,shared line @ switzerland,123,115,14,45,64,0.366,10,4,0.913,"[0.861, 0.957]",0.500,"[0.125, 0.875]",0.984
5,sex-specific lines @ switzerland,123,115,11,48,64,0.390,7,4,0.939,"[0.896, 0.983]",0.500,"[0.125, 0.875]",0.984
6,shared line @ long_beach_va,199,148,19,66,114,0.332,5,37,0.966,"[0.932, 0.993]",0.275,"[0.157, 0.392]",0.842
7,sex-specific lines @ long_beach_va,199,148,23,62,114,0.312,11,39,0.926,"[0.878, 0.966]",0.235,"[0.118, 0.353]",0.842
8,shared line @ all 3 unseen hospitals,615,369,177,187,251,0.304,29,98,0.921,"[0.892, 0.949]",0.602,"[0.541, 0.663]",0.853
9,sex-specific lines @ all 3 unseen hospitals,615,369,165,199,251,0.324,31,112,0.916,"[0.889, 0.943]",0.545,"[0.484, 0.606]",0.853


=== SEX & AGE SUBGROUP FAIRNESS AUDIT ===


,cohort,rule,group,n,disease_cases,sensitivity,sensitivity_95ci,specificity,missed_cases_FN,false_alarms_FP
0,cleveland (OOF),shared line,women,97,25,0.800,"[0.64, 0.92]",0.750,5,18
1,cleveland (OOF),shared line,men,206,114,0.930,"[0.877, 0.974]",0.391,8,56
2,cleveland (OOF),shared line,age < 55,143,44,0.841,"[0.727, 0.932]",0.687,7,31
3,cleveland (OOF),shared line,age >= 55,160,95,0.937,"[0.874, 0.979]",0.338,6,43
4,cleveland (OOF),sex-specific lines,women,97,25,0.880,"[0.72, 1.0]",0.306,3,50
5,cleveland (OOF),sex-specific lines,men,206,114,0.904,"[0.842, 0.956]",0.424,11,53
6,cleveland (OOF),sex-specific lines,age < 55,143,44,0.818,"[0.705, 0.932]",0.596,8,40
7,cleveland (OOF),sex-specific lines,age >= 55,160,95,0.937,"[0.884, 0.979]",0.031,6,63
8,hungary,shared line,women,80,12,0.333,"[0.083, 0.583]",0.882,8,8
9,hungary,shared line,men,213,94,0.936,"[0.883, 0.979]",0.588,6,49
